# Chicago Traffic Crash Analysis (2018–2025)
## Notebook 1: Data collection and cleaning

**Author:** Armon Saeedvafa
**Date:** November 2026

This is the first of two notebooks. It:

1. downloads reported crashes, person-level injury records, the street network, traffic volumes, mobility data, community-area boundaries and population estimates from the City of Chicago and the Illinois Department of Transportation (IDOT);
2. checks data quality and documents each cleaning decision;
3. builds a road-segment **exposure model** that estimates how many vehicle-miles are driven on every street block; and
4. saves the prepared datasets to `data/processed/prepared_data.pkl`.

**Run this notebook first.** [Notebook 2 (`02_analysis.ipynb`)](02_analysis.ipynb) loads the saved file and contains the executive summary, analysis, maps, findings and recommendations.

### Project folders

| Folder | Contents | Uploaded to GitHub? |
|---|---|---|
| `data/raw/` | Files downloaded by this notebook (reused on later runs) | No (too large; anyone can recreate them by running this notebook) |
| `data/processed/` | `prepared_data.pkl`, the hand-off to Notebook 2 | No |
| `images/`, `docs/maps/` | Charts and interactive maps written by Notebook 2 | Yes |

## 1. Setup and data sources

The notebook uses Chicago's Traffic Crashes dataset, the associated Traffic Crashes – People dataset, official community-area boundaries, and community-area population estimates. For exposure, it also uses:

| Exposure measure | Source | What it represents |
|---|---|---|
| Street network | City of Chicago street centerlines | Every street block, with road class, address range, length and intersection nodes |
| Traffic volume | IDOT Annual Average Daily Traffic (AADT) segments | Two-way daily traffic on counted roads |
| Traffic volume | City of Chicago camera-based Average Daily Traffic counts (Oct 2020 onward) | Directional daily traffic near ~800 traffic and speed cameras |
| Time-of-week travel pattern | Transportation Network Provider (ride-hail) trips, 2019 and 2024 samples | Share of vehicle travel by hour of the week and by community area |
| Pedestrian activity | CTA 'L' station entries and bus-stop boardings | People walking to and from transit |
| Cycling activity | Divvy bike-share trips (2018–2019) | Bike trips by station and hour |

The crash dataset is treated as crash-level data. The People dataset contains person-level records, so it must be aggregated to the crash level before being joined to crash-level data. Community boundaries support spatial assignment, population estimates support a per-resident comparison, and the exposure data support per-trip (per-vehicle-mile) comparisons.

### Reproducibility

The download cells save each calendar year's data, and each exposure dataset, as a local file. On later runs, existing files are reused instead of downloaded again. Everything is saved under `data/raw/`. To refresh the data from the City or IDOT APIs, remove the relevant saved files first (crash and People files in `data/raw/chicago_crash_data/` and `data/raw/chicago_people_data/`; exposure, boundary and population files in `data/raw/chicago_exposure_data/`).

The required packages are listed in `requirements.txt` (install them with `pip install -r requirements.txt`). The first full run downloads several large datasets, so it can take a while; later runs reuse the saved files.

In [15]:
# Setup: libraries, display options, study-period constants and project folders.
import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from shapely.geometry import LineString

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

START_YEAR = 2018
END_YEAR = 2025
N_YEARS = END_YEAR - START_YEAR + 1

# Project folders. The notebook can be run from the notebooks/ folder or from the project's main folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"              # downloaded files (reused on later runs)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"  # prepared data handed to Notebook 2
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project folder: {PROJECT_ROOT.resolve()}")

Project folder: /Users/armon


## 2. Collect and understand the data

### 2.1 Download or load the crash-level dataset

This retrieves selected fields for 2018–2025. Each year is saved separately so later runs can reuse completed downloads.

In [16]:
# Shared download helpers, then download (or load) the Traffic Crashes dataset, one calendar year at a time.
SOCRATA_BASE = "https://data.cityofchicago.org/resource"
HTTP = requests.Session()
HTTP.headers.update({"User-Agent": "ChicagoCrashCapstone/1.0"})


def request_with_retries(url, params=None, timeout=(20, 300), max_retries=5):
    """GET a URL, retrying on rate limits, server errors and dropped connections."""
    for attempt in range(max_retries):
        wait_seconds = min(5 * (2 ** attempt), 60)
        try:
            response = HTTP.get(url, params=params, timeout=timeout)
            if response.status_code == 400:  # an invalid query will not succeed on retry
                print("Invalid API query:", response.text[:2000])
                response.raise_for_status()
            if response.status_code == 429 or response.status_code >= 500:
                if attempt == max_retries - 1:
                    response.raise_for_status()
                print(f"API error {response.status_code}; retrying in {wait_seconds} seconds.")
                time.sleep(wait_seconds)
                continue
            response.raise_for_status()
            return response
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as error:
            if attempt == max_retries - 1:
                raise RuntimeError(f"Request failed after {max_retries} attempts: {url}") from error
            print(f"Connection issue; retrying in {wait_seconds} seconds.")
            time.sleep(wait_seconds)


def socrata_all(dataset_id, params, page_size=50000, show_progress=False):
    """Page through a City of Chicago (Socrata) query and return every row as a DataFrame."""
    pages, offset = [], 0
    while True:
        batch = request_with_retries(f"{SOCRATA_BASE}/{dataset_id}.json",
                                     dict(params, **{"$limit": page_size, "$offset": offset})).json()
        if not batch:
            break
        pages.append(pd.DataFrame(batch))
        offset += len(batch)
        if show_progress:
            print(f"Retrieved {offset:,} records for this year.")
        if len(batch) < page_size:
            break
        time.sleep(0.5)
    return pd.concat(pages, ignore_index=True) if pages else pd.DataFrame()


def load_years(dataset_id, columns, folder, prefix, label, page_size, dtypes):
    """Load each year's saved CSV, or download and save that year if it is not saved yet."""
    folder.mkdir(parents=True, exist_ok=True)
    years = []
    for year in range(START_YEAR, END_YEAR + 1):
        path = folder / f"{prefix}_{year}.csv"
        if path.exists():
            print(f"\nLoading saved {label} for {year}...")
            data = pd.read_csv(path, dtype=dtypes, low_memory=False)
        else:
            print(f"\nDownloading {label} for {year}...")
            data = socrata_all(dataset_id, {
                "$select": ",".join(columns),
                "$where": f"crash_date >= '{year}-01-01T00:00:00' AND crash_date < '{year + 1}-01-01T00:00:00'",
                "$order": "crash_date ASC, crash_record_id ASC",
            }, page_size=page_size, show_progress=True)
            data.to_csv(path, index=False)  # saved only after the whole year downloads
            print(f"Saved {len(data):,} records to {path}")
        years.append(data)
    return pd.concat(years, ignore_index=True)


# 1. Fields kept from the crash dataset.
COLUMNS = [
    "crash_record_id", "crash_date", "posted_speed_limit", "traffic_control_device", "device_condition",
    "weather_condition", "lighting_condition", "first_crash_type", "trafficway_type", "roadway_surface_cond",
    "road_defect", "intersection_related_i", "hit_and_run_i", "crash_type", "damage", "prim_contributory_cause",
    "num_units", "most_severe_injury", "injuries_total", "injuries_fatal", "injuries_incapacitating",
    "injuries_non_incapacitating", "injuries_reported_not_evident", "latitude", "longitude", "street_name",
    "street_no", "street_direction", "beat_of_occurrence", "location",
]
CRASH_FOLDER = RAW_DIR / "chicago_crash_data"

# 2. Load or download 2018–2025 and combine the years.
df = load_years("85ca-t3if", COLUMNS, CRASH_FOLDER, "crashes", "crash data", page_size=10000,
                dtypes={"crash_record_id": "string"})

# 3. Standardize types: text IDs, numeric counts and coordinates, and timestamps.
df["crash_record_id"] = df["crash_record_id"].astype("string")
numeric_columns = ["posted_speed_limit", "injuries_total", "injuries_fatal", "injuries_incapacitating",
                   "injuries_non_incapacitating", "injuries_reported_not_evident", "latitude", "longitude", "num_units"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")

print("\nCrash-data loading complete.")
print(f"Total rows: {len(df):,}")
print(f"Unique crash IDs: {df['crash_record_id'].nunique():,}")
print(f"Date range: {df['crash_date'].min()} to {df['crash_date'].max()}")
print(f"Local CSV folder: {CRASH_FOLDER.resolve()}")


Loading saved crash data for 2018...

Loading saved crash data for 2019...

Loading saved crash data for 2020...

Loading saved crash data for 2021...

Loading saved crash data for 2022...

Loading saved crash data for 2023...

Loading saved crash data for 2024...

Loading saved crash data for 2025...

Crash-data loading complete.
Total rows: 877,919
Unique crash IDs: 877,919
Date range: 2018-01-01 00:00:00 to 2025-12-31 23:52:00
Local CSV folder: /Users/armon/data/raw/chicago_crash_data


### 2.2 Download or load the People dataset

The People dataset contains person-level injury classifications. These records will be aggregated to one row per crash before being joined to crash-level data.

In [17]:
# Download (or load) the Traffic Crashes – People dataset (one row per person involved), one year at a time.
PEOPLE_COLUMNS = ["person_id", "crash_record_id", "crash_date", "person_type", "injury_classification", "vehicle_id"]
id_columns = ["person_id", "crash_record_id", "vehicle_id"]

people = load_years("u6pd-qa9d", PEOPLE_COLUMNS, RAW_DIR / "chicago_people_data", "people", "People records",
                    page_size=5000, dtypes={col: "string" for col in id_columns})
people[id_columns] = people[id_columns].astype("string")

print("\nDownload complete.")
print("People records:", f"{len(people):,}")
print("Unique person IDs:", f"{people['person_id'].nunique():,}")


Loading saved People records for 2018...

Loading saved People records for 2019...

Loading saved People records for 2020...

Loading saved People records for 2021...

Loading saved People records for 2022...

Loading saved People records for 2023...

Loading saved People records for 2024...

Loading saved People records for 2025...

Download complete.
People records: 1,927,651
Unique person IDs: 1,927,651


### 2.3 Review source categories

Inspect the date coverage and the actual injury classification labels before defining serious outcomes. This helps prevent category-label assumptions from silently affecting the results.

In [18]:
# Check the date coverage and the injury and person-type labels before defining serious outcomes.
print("Crash date range:")
print(df["crash_date"].agg(["min", "max"]))
print("\nInjury classifications:")
display(people["injury_classification"].value_counts(dropna=False))
print("\nPerson types:")
display(people["person_type"].value_counts(dropna=False))

Crash date range:
min   2018-01-01 00:00:00
max   2025-12-31 23:52:00
Name: crash_date, dtype: datetime64[us]

Injury classifications:


injury_classification
NO INDICATION OF INJURY     1742917
NONINCAPACITATING INJURY     103093
REPORTED, NOT EVIDENT         62223
INCAPACITATING INJURY         17779
FATAL                          1083
NaN                             556
Name: count, dtype: int64


Person types:


person_type
DRIVER                 1494890
PASSENGER               391030
PEDESTRIAN               24224
BICYCLE                  15520
NON-MOTOR VEHICLE         1687
NON-CONTACT VEHICLE        300
Name: count, dtype: int64

### 2.4 Download or load road-network and traffic-volume data

Crash counts alone cannot show whether a place is dangerous or simply busy. To estimate **vehicle-miles travelled (VMT)**, the notebook combines:

- **Street centerlines:** every street block in Chicago, used as the unit for exposure and for matching crashes to streets and intersections.
- **IDOT AADT:** Illinois DOT's two-way annual average daily traffic on counted road segments (count years vary by segment).
- **City camera ADT:** the City's directional 30-day-average traffic counts from traffic and speed cameras (October 2020 onward), averaged over the available period for each segment.

In [19]:
# Download (or load) the road network and traffic-volume data. Each file is saved after its first download.
EXPOSURE_FOLDER = RAW_DIR / "chicago_exposure_data"
EXPOSURE_FOLDER.mkdir(exist_ok=True)


def cached(name, loader, geo=False):
    """Load EXPOSURE_FOLDER/<name> if it was saved before; otherwise run loader() and save the result."""
    path = EXPOSURE_FOLDER / f"{name}.{'geojson' if geo else 'csv'}"
    if path.exists():
        print(f"Loading saved {name}: {path}")
        return gpd.read_file(path) if geo else pd.read_csv(path, low_memory=False)
    print(f"Downloading {name}...")
    data = loader()
    data.to_file(path, driver="GeoJSON") if geo else data.to_csv(path, index=False)
    print(f"Saved {len(data):,} {'features' if geo else 'rows'} to {path}")
    return data


def paged_geojson(url, params, limit_key, offset_key, label, page_size):
    """Page through a GeoJSON web service and return all features as one GeoDataFrame."""
    pages, offset = [], 0
    while True:
        payload = request_with_retries(url, dict(params, **{limit_key: page_size, offset_key: offset})).json()
        if "error" in payload:
            raise RuntimeError(f"Service error: {payload['error']}")
        features = payload.get("features", [])
        if not features:
            break
        pages.append(gpd.GeoDataFrame.from_features(features, crs="EPSG:4326"))
        offset += len(features)
        print(f"Retrieved {offset:,} {label}")
        if len(features) < page_size:
            break
    return gpd.GeoDataFrame(pd.concat(pages, ignore_index=True), geometry="geometry", crs="EPSG:4326")


# 1. City street centerlines: every street block, with road class, address range and intersection node IDs.
centerlines = cached("street_centerlines", lambda: paged_geojson(
    f"{SOCRATA_BASE}/pr57-gg9e.geojson",
    {"$select": "objectid,fnode_id,tnode_id,pre_dir,street_nam,street_typ,suf_dir,"
                "l_f_add,l_t_add,r_f_add,r_t_add,class,the_geom", "$order": "objectid"},
    "$limit", "$offset", "centerline segments", page_size=50000), geo=True)

# 2. IDOT annual average daily traffic (two-way volumes) on road segments inside the Chicago screening box.
idot_aadt = cached("idot_aadt_chicago", lambda: paged_geojson(
    "https://services2.arcgis.com/aIrBD8yn1TDTEXoz/ArcGIS/rest/services/ADT_COMP_2020Dissolved/FeatureServer/0/query",
    {"where": "COUNTY_NAM='COOK' AND AADT > 0", "geometry": "-87.95,41.64,-87.52,42.03",
     "geometryType": "esriGeometryEnvelope", "inSR": 4326, "spatialRel": "esriSpatialRelIntersects",
     "outFields": "OBJECTID,AADT,AADT_YR", "outSR": 4326, "returnGeometry": "true",
     "orderByFields": "OBJECTID", "f": "geojson"},
    "resultRecordCount", "resultOffset", "IDOT AADT segments", page_size=1000), geo=True)
idot_aadt[["AADT", "AADT_YR"]] = idot_aadt[["AADT", "AADT_YR"]].apply(pd.to_numeric, errors="coerce")

# 3. City camera-based traffic counts (directional, Oct 2020 onward), averaged per segment on the City's server.
#    Aliases differ from column names so SoQL does not confuse them with the source columns.
camera_adt = cached("camera_adt_by_segment", lambda: socrata_all("gc7y-n4xa", {
    "$select": ("segmentid, roadname, direction, fromsegment, tosegment, avg(startlat) as start_lat, "
                "avg(startlon) as start_lon, avg(endlat) as end_lat, avg(endlon) as end_lon, "
                "avg(vehiclecount) as adt, count(*) as days"),
    "$where": f"vehiclecount > 0 AND timestamp < '{END_YEAR + 1}-01-01T00:00:00'",
    "$group": "segmentid, roadname, direction, fromsegment, tosegment",
    "$order": "segmentid",
}).rename(columns={"start_lat": "startlat", "start_lon": "startlon", "end_lat": "endlat", "end_lon": "endlon"}))
camera_cols = ["startlat", "startlon", "endlat", "endlon", "adt", "days"]
camera_adt[camera_cols] = camera_adt[camera_cols].apply(pd.to_numeric, errors="coerce")

print(f"\nStreet centerline segments: {len(centerlines):,}")
print(f"IDOT AADT segments in the Chicago screening box: {len(idot_aadt):,} "
      f"(count years {int(idot_aadt['AADT_YR'].min())}–{int(idot_aadt['AADT_YR'].max())})")
print(f"Camera ADT directional segments: {len(camera_adt):,} "
      f"(median {camera_adt['days'].median():.0f} days of counts per segment)")

Loading saved street_centerlines: /Users/armon/data/raw/chicago_exposure_data/street_centerlines.geojson
Loading saved idot_aadt_chicago: /Users/armon/data/raw/chicago_exposure_data/idot_aadt_chicago.geojson
Loading saved camera_adt_by_segment: /Users/armon/data/raw/chicago_exposure_data/camera_adt_by_segment.csv

Street centerline segments: 56,338
IDOT AADT segments in the Chicago screening box: 4,207 (count years 2016–2025)
Camera ADT directional segments: 2,269 (median 793 days of counts per segment)


### 2.5 Download or load mobility and road-user activity data

Traffic counts give *daily* volumes. To estimate *when* people travel, and to measure walking and cycling activity, the notebook adds:

- **Ride-hail (TNP) trips** aggregated on the City's server by pickup community area and hour. One full week in each season (the first seven days of January, April, July and October) of 2019 and 2024 is used: 56 days and about 14.4 million trips. These trips are used only for the *shape* of travel across the hours of the week; traffic volumes come from the traffic counts. They show how vehicle travel is spread across the week and how that pattern differs between areas.
- **CTA 'L' station entries** (average daily entries per station, 2018–2025) and **bus-stop boardings** (average weekday boardings per stop), used as an indicator of pedestrian activity.
- **Divvy bike-share trips** (2018–2019, the years available on the City portal) by start station and by hour, used as an indicator of cycling activity.

These are *indicators* of activity, not complete counts of every driver, pedestrian or cyclist. Notebook 2 (Section 9) discusses their limitations.

In [20]:
# Download (or load) mobility and road-user activity data used as exposure measures:
#   - ride-hail trips: how vehicle travel is spread across the hours of the week, overall and by area;
#   - CTA 'L' station entries and bus-stop boardings: an indicator of pedestrian activity around transit;
#   - Divvy bike-share trips: an indicator of cycling activity.
# The large ride-hail and bike-share tables are summarised on the City's server in small pieces (one day or one
# month per request), several at once. Each piece is saved in data/raw/chicago_exposure_data/download_chunks, so an
# interrupted run resumes where it stopped.
CHUNK_FOLDER = EXPOSURE_FOLDER / "download_chunks"
CHUNK_FOLDER.mkdir(exist_ok=True)
PARALLEL_REQUESTS = 6
_thread_state = threading.local()


def cached_chunk(name, dataset_id, params, max_retries=8):
    """Run one small summary query (with its own HTTP session per thread) and save the result."""
    path = CHUNK_FOLDER / f"{name}.csv"
    if path.exists():
        return pd.read_csv(path, low_memory=False)
    if not hasattr(_thread_state, "session"):
        _thread_state.session = requests.Session()
        _thread_state.session.headers.update(HTTP.headers)
    for attempt in range(max_retries):
        try:
            response = _thread_state.session.get(f"{SOCRATA_BASE}/{dataset_id}.json",
                                                 params=dict(params, **{"$limit": 50000}), timeout=(20, 600))
            if response.status_code == 400:
                raise ValueError(f"Invalid API query for {name}: {response.text[:1000]}")
            if response.status_code == 429 or response.status_code >= 500:
                raise requests.exceptions.ConnectionError(f"HTTP {response.status_code}")
            response.raise_for_status()
            part = pd.DataFrame(response.json())
            part.to_csv(path, index=False)
            return part
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt == max_retries - 1:
                raise RuntimeError(f"{name}: failed after {max_retries} attempts. Rerun the cell to resume.")
            time.sleep(min(5 * (2 ** attempt), 120))


def fetch_chunks(jobs, label):
    """Download many (name, dataset, query) pieces in parallel, reusing saved pieces; returns {name: DataFrame}."""
    saved = sum((CHUNK_FOLDER / f"{name}.csv").exists() for name, _, _ in jobs)
    print(f"{label}: {saved:,} of {len(jobs):,} pieces already saved; downloading {len(jobs) - saved:,} "
          f"({PARALLEL_REQUESTS} at a time)...")
    results, started = {}, time.time()
    with ThreadPoolExecutor(max_workers=PARALLEL_REQUESTS) as pool:
        futures = {pool.submit(cached_chunk, *job): job[0] for job in jobs}
        for done, future in enumerate(as_completed(futures), start=1):
            results[futures[future]] = future.result()
            if done % 10 == 0 or done == len(jobs):
                print(f"  {label}: {done:,}/{len(jobs):,} pieces ready ({(time.time() - started) / 60:.1f} min elapsed)")
    return results


def add_lat_lon(frame, column="location"):
    """Split a Socrata point column into lat/lon (some City datasets store [lat, lon], others [lon, lat])."""
    def parse(value):
        if isinstance(value, str):
            try:
                value = json.loads(value.replace("'", '"'))
            except ValueError:
                return np.nan, np.nan
        if isinstance(value, dict) and "coordinates" in value:
            a, b = (float(v) for v in value["coordinates"][:2])
            return (a, b) if 40 < a < 43 else (b, a)
        if isinstance(value, dict) and "latitude" in value:
            return float(value["latitude"]), float(value["longitude"])
        return np.nan, np.nan
    frame[["lat", "lon"]] = frame[column].apply(parse).tolist()
    return frame.drop(columns=column)


# 1. Ride-hail trips by pickup community area and hour, one day per request. The sample is one full week in each
#    season (1–7 January, April, July and October) of 2019 (pre-pandemic) and 2024 (most recent full year):
#    56 days, each weekday appearing 8 times. Traffic volumes themselves come from the counts in code cell 5.
TNP_SOURCES = {2019: "m6dm-c72p", 2024: "n26f-ihde"}


def load_tnp_profile():
    days = [(year, dataset_id, day) for year, dataset_id in TNP_SOURCES.items() for month in [1, 4, 7, 10]
            for day in pd.date_range(f"{year}-{month:02d}-01", periods=7, freq="D")]
    pieces = fetch_chunks([(f"tnp_{day:%Y-%m-%d}", dataset_id, {
        "$select": ("pickup_community_area, date_extract_hh(trip_start_timestamp) as hour, "
                    "count(*) as trip_count, sum(trip_miles) as trip_miles_sum"),
        "$where": (f"trip_start_timestamp >= '{day:%Y-%m-%dT%H:%M:%S}' AND trip_start_timestamp < "
                   f"'{day + pd.Timedelta(days=1):%Y-%m-%dT%H:%M:%S}' AND trip_miles > 0 AND trip_miles < 100"),
        "$group": "pickup_community_area, date_extract_hh(trip_start_timestamp)",
    }) for year, dataset_id, day in days], "Ride-hail days")
    return pd.concat([
        pieces[f"tnp_{day:%Y-%m-%d}"].rename(columns={"trip_count": "trips", "trip_miles_sum": "trip_miles"})
        .assign(year=year, date=day.date(), day_index=day.dayofweek)  # day_index: Monday = 0
        for year, _, day in days], ignore_index=True)


tnp = cached("tnp_hourly_profile_seasonal_weeks", load_tnp_profile)
tnp_cols = ["pickup_community_area", "hour", "trips", "trip_miles", "day_index", "year"]
tnp[tnp_cols] = tnp[tnp_cols].apply(pd.to_numeric, errors="coerce")
tnp = tnp.dropna(subset=["day_index", "hour"]).astype({"day_index": int, "hour": int})

# 2. CTA 'L' stations: locations, and average daily entries per station over 2018–2025.
l_stops = cached("cta_l_stops", lambda: add_lat_lon(socrata_all("8pix-ypme", {
    "$select": "map_id, location", "$order": "map_id"})))
l_entries = cached("cta_l_station_entries", lambda: socrata_all("5neh-572f", {
    "$select": "station_id, avg(rides) as avg_daily_entries",
    "$where": f"date >= '{START_YEAR}-01-01T00:00:00' AND date < '{END_YEAR + 1}-01-01T00:00:00'",
    "$group": "station_id", "$order": "station_id"}))
l_stations = (l_stops.groupby("map_id", as_index=False)[["lat", "lon"]].mean()
              .merge(l_entries.rename(columns={"station_id": "map_id"}), on="map_id"))
l_stations["avg_daily_entries"] = pd.to_numeric(l_stations["avg_daily_entries"], errors="coerce")

# 3. CTA bus stops: average weekday boardings per stop (the City's stop-level release).
bus_stops = cached("cta_bus_stop_boardings", lambda: add_lat_lon(socrata_all("mq3i-nnqe", {
    "$select": "stop_id, boardings, location", "$order": "stop_id"})))
bus_stops["boardings"] = pd.to_numeric(bus_stops["boardings"], errors="coerce")

# 4. Divvy bike-share trips by start station and by hour of day, one month per request. The City portal holds
#    Divvy trips only up to 2019, so 2018–2019 is the only period that overlaps the crash data.
DIVVY_MONTHS = pd.date_range("2018-01-01", "2019-12-01", freq="MS")


def divvy_chunks(prefix, select, group):
    return pd.concat(fetch_chunks([(f"{prefix}_{m:%Y-%m}", "fg6s-gzvg", {
        "$select": select, "$group": group,
        "$where": f"start_time >= '{m:%Y-%m-%dT%H:%M:%S}' AND "
                  f"start_time < '{m + pd.offsets.MonthBegin(1):%Y-%m-%dT%H:%M:%S}'"}) for m in DIVVY_MONTHS],
        f"Divvy {prefix.split('_')[1]} months").values(), ignore_index=True).apply(pd.to_numeric, errors="coerce")


divvy_stations = cached("divvy_trips_by_station", lambda: divvy_chunks(
    "divvy_stations", "from_station_id, avg(from_latitude) as station_lat, avg(from_longitude) as station_lon, "
    "count(*) as trip_count", "from_station_id")
    .groupby("from_station_id", as_index=False)
    .agg(lat=("station_lat", "mean"), lon=("station_lon", "mean"), trips=("trip_count", "sum")))
divvy_hourly = cached("divvy_trips_by_hour", lambda: divvy_chunks(
    "divvy_hourly", "date_extract_hh(start_time) as hour, count(*) as trip_count", "date_extract_hh(start_time)")
    .groupby("hour", as_index=False)["trip_count"].sum().rename(columns={"trip_count": "trips"}))

print(f"\nRide-hail sample: {tnp['trips'].sum():,.0f} trips and {tnp['trip_miles'].sum():,.0f} trip-miles "
      f"({', '.join(map(str, TNP_SOURCES))}; {tnp['date'].nunique():,} sampled days)")
print(f"CTA 'L' stations with ridership: {len(l_stations):,}; "
      f"average daily entries citywide: {l_stations['avg_daily_entries'].sum():,.0f}")
print(f"CTA bus stops with boardings: {bus_stops['boardings'].notna().sum():,}; "
      f"average weekday boardings: {bus_stops['boardings'].sum():,.0f}")
print(f"Divvy stations: {len(divvy_stations):,}; trips 2018–2019: {divvy_stations['trips'].sum():,.0f}")

Loading saved tnp_hourly_profile_seasonal_weeks: /Users/armon/data/raw/chicago_exposure_data/tnp_hourly_profile_seasonal_weeks.csv
Loading saved cta_l_stops: /Users/armon/data/raw/chicago_exposure_data/cta_l_stops.csv
Loading saved cta_l_station_entries: /Users/armon/data/raw/chicago_exposure_data/cta_l_station_entries.csv
Loading saved cta_bus_stop_boardings: /Users/armon/data/raw/chicago_exposure_data/cta_bus_stop_boardings.csv
Loading saved divvy_trips_by_station: /Users/armon/data/raw/chicago_exposure_data/divvy_trips_by_station.csv
Loading saved divvy_trips_by_hour: /Users/armon/data/raw/chicago_exposure_data/divvy_trips_by_hour.csv

Ride-hail sample: 14,416,588 trips and 94,343,396 trip-miles (2019, 2024; 56 sampled days)
CTA 'L' stations with ridership: 144; average daily entries citywide: 310,543
CTA bus stops with boardings: 11,593; average weekday boardings: 1,054,478
Divvy stations: 618; trips 2018–2019: 7,420,746


### 2.6 Download or load community-area boundaries and population

Chicago is divided into 77 official **community areas**. Their boundaries are used to assign each crash to an area, and the 2023 population estimates support a per-resident comparison in Notebook 2. Both files are saved after the first download.

In [21]:
# Load the official community-area boundaries and 2023 population estimates (saved after the first download).
BOUNDARY_URL = "https://data.cityofchicago.org/api/v3/views/igwz-8jzy/query.geojson?accessType=DOWNLOAD"
community_areas = cached("community_area_boundaries", lambda: gpd.read_file(BOUNDARY_URL), geo=True)

# Standardize the name and number fields so they match across datasets.
community_areas = community_areas.rename(columns={"community": "community_name", "area_numbe": "community_number"})
community_areas["community_name"] = community_areas["community_name"].astype("string").str.strip().str.upper()
community_areas["community_number"] = pd.to_numeric(community_areas["community_number"], errors="coerce")
community_areas = community_areas.to_crs("EPSG:4326")

# Population: keep the 2023 estimate for each area.
population_raw = cached("community_area_population",
                        lambda: pd.DataFrame(request_with_retries(f"{SOCRATA_BASE}/t68z-cikk.json",
                                                                  {"$limit": 500}).json()))
population = (population_raw.loc[population_raw["acs_year"].astype(str).str.strip().eq("2023"),
                                 ["community_area", "total_population"]]
              .rename(columns={"community_area": "community_name", "total_population": "population"}))
population["community_name"] = population["community_name"].astype("string").str.strip().str.upper()
population["population"] = pd.to_numeric(population["population"], errors="coerce")
population = population.loc[population["population"].gt(0)]

print(f"Community areas in boundary file: {len(community_areas)}")
print(f"Community areas with a 2023 population estimate: {len(population)}")

Loading saved community_area_boundaries: /Users/armon/data/raw/chicago_exposure_data/community_area_boundaries.geojson
Loading saved community_area_population: /Users/armon/data/raw/chicago_exposure_data/community_area_population.csv
Community areas in boundary file: 77
Community areas with a 2023 population estimate: 77


## 3. Data quality and preparation

### 3.1 Normalize fields and audit crash IDs

The original downloaded crash data is preserved in `df`. Repeated IDs are audited before reducing the data to one row per crash for analysis.

In [22]:
# Audit repeated crash IDs: count them and check whether any repeated ID has conflicting key fields.
duplicate_mask = df["crash_record_id"].duplicated(keep=False)
audit_fields = ["crash_date", "latitude", "longitude", "injuries_total", "injuries_fatal",
                "injuries_incapacitating", "most_severe_injury"]
conflicting_ids = (df.loc[duplicate_mask].groupby("crash_record_id")[audit_fields]
                   .nunique(dropna=False).gt(1).any(axis=1).sum())

print(f"Downloaded rows: {len(df):,}")
print(f"Rows flagged for repeated crash IDs: {int(duplicate_mask.sum()):,}")
print(f"Repeated IDs with conflicting key fields: {int(conflicting_ids):,}")

Downloaded rows: 877,919
Rows flagged for repeated crash IDs: 0
Repeated IDs with conflicting key fields: 0


### 3.2 Document cleaning decisions

Missing dates prevent time-based analysis. Missing or invalid coordinates prevent spatial analysis but do not automatically exclude a crash from non-geographic analysis. The broad geographic bounds below are a screening rule, not an official city boundary.

In [23]:
# Summarise data-quality issues and the cleaning decision taken for each.
in_bounds = df["latitude"].between(41.60, 42.10) & df["longitude"].between(-87.95, -87.50)
has_coordinates = df["latitude"].notna() & df["longitude"].notna()

audit = pd.DataFrame([
    ("Repeated crash IDs", duplicate_mask.sum(),
     "Inspect key-field conflicts; count each crash ID once in summaries."),
    ("Missing or invalid crash date", df["crash_date"].isna().sum(), "Exclude from time-based analyses."),
    ("Missing latitude or longitude", (~has_coordinates).sum(),
     "Retain for non-geographic analyses; exclude from spatial joins."),
    ("Outside broad Chicago screening bounds", (has_coordinates & ~in_bounds).sum(),
     "Review; exclude from spatial analysis unless verified."),
    ("Zero latitude or longitude", (df["latitude"].eq(0) | df["longitude"].eq(0)).sum(),
     "Treat as invalid for geographic analysis."),
], columns=["issue", "rows_flagged", "decision"]).astype({"rows_flagged": int})
display(audit)

,issue,rows_flagged,decision
0,Repeated crash IDs,0,Inspect key-field conflicts; count each crash ...
1,Missing or invalid crash date,0,Exclude from time-based analyses.
2,Missing latitude or longitude,7075,Retain for non-geographic analyses; exclude fr...
3,Outside broad Chicago screening bounds,55,Review; exclude from spatial analysis unless v...
4,Zero latitude or longitude,55,Treat as invalid for geographic analysis.


### 3.3 Create analysis-ready crash and geographic datasets

The temporal dataset keeps records with valid dates. For crash-level counting, one row per crash ID is used after the duplicate audit.

In [24]:
# Build the analysis datasets: one row per crash with a valid date, and the subset with valid coordinates.
df_clean = (df.loc[df["crash_date"].notna()]
            .sort_values(["crash_record_id", "crash_date"])
            .drop_duplicates(subset=["crash_record_id"], keep="first"))
valid_coordinates = (df_clean["latitude"].between(41.60, 42.10) & df_clean["longitude"].between(-87.95, -87.50)
                     & df_clean["latitude"].ne(0) & df_clean["longitude"].ne(0))
df_geo = df_clean.loc[valid_coordinates].copy()

print(f"Raw rows: {len(df):,}")
print(f"Crash-level records with valid dates: {len(df_clean):,}")
print(f"Records eligible for geographic analysis: {len(df_geo):,}")
print(f"Date range used: {df_clean['crash_date'].min()} to {df_clean['crash_date'].max()}")

Raw rows: 877,919
Crash-level records with valid dates: 877,919
Records eligible for geographic analysis: 870,789
Date range used: 2018-01-01 00:00:00 to 2025-12-31 23:52:00


### 3.4 Prepare person-level injury information

Exact duplicate People rows are removed. Person IDs associated with multiple crash IDs or conflicting injury classifications are flagged for review rather than silently resolved by keeping the first record.

For this project, a **serious-outcome crash** is a crash involving at least one person classified as `FATAL` or `INCAPACITATING INJURY`. A crash outcome is considered known if at least one person has a non-missing classification other than `UNKNOWN`.

In [25]:
# Standardize People labels, remove exact duplicate rows, and flag person IDs with inconsistent records.
for col in ["person_type", "injury_classification"]:
    people[col] = people[col].astype("string").str.strip().str.upper()

people_rows_before = len(people)
people = people.drop_duplicates().reset_index(drop=True)

person_review = people.dropna(subset=["person_id"]).groupby("person_id").agg(
    distinct_crash_ids=("crash_record_id", "nunique"),
    distinct_injury_classes=("injury_classification", "nunique"),
    distinct_person_types=("person_type", "nunique"),
    rows=("person_id", "size"),
)
flagged_people = (person_review[["distinct_crash_ids", "distinct_injury_classes", "distinct_person_types", "rows"]]
                  .gt(1).any(axis=1).sum())

print(f"People rows before removing exact duplicates: {people_rows_before:,}")
print(f"Exact duplicate People rows removed: {people_rows_before - len(people):,}")
print(f"People rows after removing exact duplicates: {len(people):,}")
print(f"Person IDs flagged for review: {flagged_people:,}")

People rows before removing exact duplicates: 1,927,651
Exact duplicate People rows removed: 0
People rows after removing exact duplicates: 1,927,651
Person IDs flagged for review: 0


### 3.5 Aggregate People records to crash level

The People dataset is aggregated before joining it to the crash table, so a crash involving multiple people is still counted once. Unknown classifications are not automatically treated as non-serious. The aggregation also records how many drivers, pedestrians and cyclists each crash involved. These flags are used for the pedestrian and cyclist exposure rates and to convert crash rates into per-driver probabilities.

In [26]:
# Summarise the People data to one row per crash and attach it to the crash table.
# A crash has a serious outcome if anyone was killed or incapacitated; its outcome is "known" if at least
# one person has a classification other than missing or UNKNOWN.
injury_class = people["injury_classification"]
people["classification_known"] = injury_class.notna() & ~injury_class.isin(["UNKNOWN", ""])
people["is_serious_person"] = injury_class.isin(["FATAL", "INCAPACITATING INJURY"])
for flag, person_type in [("is_driver", "DRIVER"), ("is_pedestrian", "PEDESTRIAN"), ("is_cyclist", "BICYCLE")]:
    people[flag] = people["person_type"].eq(person_type)

people_by_crash = people.dropna(subset=["crash_record_id"]).groupby("crash_record_id").agg(
    has_known_outcome=("classification_known", "any"),
    has_serious_outcome=("is_serious_person", "any"),
    drivers=("is_driver", "sum"),
    pedestrian_involved=("is_pedestrian", "any"),
    cyclist_involved=("is_cyclist", "any"),
).reset_index()

crash_enriched = df_clean.merge(people_by_crash, on="crash_record_id", how="left", validate="one_to_one")
# serious_outcome is True/False when the outcome is known, and missing (NA) when it is not.
known = crash_enriched["has_known_outcome"].fillna(False).astype(bool)
crash_enriched["serious_outcome"] = (crash_enriched["has_serious_outcome"].fillna(False).astype(bool)
                                     .astype("boolean").where(known))
crash_enriched[["pedestrian_involved", "cyclist_involved"]] = (
    crash_enriched[["pedestrian_involved", "cyclist_involved"]].fillna(False).astype(bool))

print(f"Crashes with known outcome status: {crash_enriched['serious_outcome'].notna().sum():,}")
print(f"Crashes involving serious outcomes: {crash_enriched['serious_outcome'].eq(True).sum():,}")
print(f"Crashes involving a pedestrian: {crash_enriched['pedestrian_involved'].sum():,}")
print(f"Crashes involving a cyclist: {crash_enriched['cyclist_involved'].sum():,}")
print(f"Average drivers recorded per crash: {crash_enriched['drivers'].mean():.2f}")

Crashes with known outcome status: 875,925
Crashes involving serious outcomes: 16,023
Crashes involving a pedestrian: 23,040
Crashes involving a cyclist: 15,360
Average drivers recorded per crash: 1.71


### 3.6 Build the road-segment exposure model

Every street block covered by the crash dataset receives a daily traffic volume, in this order of preference:

1. **City camera count:** directional counts on the same street within 15 m, added together for a two-way volume.
2. **IDOT AADT:** the nearest IDOT count segment within 15 m.
3. **Estimate:** the median measured volume for the block's road class (arterial, collector, local street, and so on).

**Vehicle-miles travelled** on each block = daily volume × block length (miles) × number of days in 2018–2025.

Interstate expressways and ramps are excluded because Chicago Police crash reports exclude them. Lake Shore Drive is kept. The coverage table shows how much of each road class has a measured volume. Rates for areas or corridors that depend mostly on estimated volumes are flagged as lower-confidence in Notebook 2.

In [27]:
# Road-segment exposure model: a daily traffic volume and vehicle-miles travelled (VMT) for every street block.
PROJECTED_CRS = "EPSG:26971"  # NAD83 / Illinois East (metres), used for distances and lengths
METERS_PER_MILE = 1609.344
STUDY_DAYS = (pd.Timestamp(f"{END_YEAR + 1}-01-01") - pd.Timestamp(f"{START_YEAR}-01-01")).days
VOLUME_MATCH_DISTANCE_M = 15  # a block takes a counted volume only if the count line is this close

# 1. Standardize street names, road classes, node IDs and address ranges.
streets = centerlines.copy()
text_cols = ["pre_dir", "street_nam", "street_typ", "suf_dir", "class"]
streets[text_cols] = streets[text_cols].apply(lambda s: s.astype("string").fillna("").str.strip().str.upper())
number_cols = ["fnode_id", "tnode_id", "l_f_add", "l_t_add", "r_f_add", "r_t_add"]
streets[number_cols] = streets[number_cols].apply(pd.to_numeric, errors="coerce")
streets["street_full"] = (streets["pre_dir"] + " " + streets["street_nam"] + " " + streets["street_typ"] + " "
                          + streets["suf_dir"]).str.split().str.join(" ")

# 2. Keep the streets covered by Chicago Police crash reports: drop interstates and ramps (outside CPD's
#    jurisdiction), airport/private ways, rivers, unclassified lines and empty geometries. Lake Shore Drive stays.
is_interstate = streets["class"].eq("1") & (streets["street_typ"].isin(["EXPY", "RL"])
                                            | streets["street_nam"].str.contains(r"SKYWAY|\bI\d", regex=True))
has_geometry = streets.geometry.apply(lambda g: g is not None and not g.is_empty)
road = streets.loc[~is_interstate & ~streets["class"].isin(["9", "99", "RIV", ""]) & has_geometry]
road = road.to_crs(PROJECTED_CRS).reset_index(drop=True)
road["segment_id"] = np.arange(len(road))
road["length_m"] = road.geometry.length
road = road.loc[road["length_m"] > 1].copy()
road["road_class"] = road["class"].map({"1": "Lake Shore Drive", "2": "Arterial", "3": "Collector",
                                        "4": "Local street", "7": "Lower-level street"}).fillna("Other street")
midpoints = gpd.GeoDataFrame(road[["segment_id", "street_nam"]],
                             geometry=road.geometry.interpolate(0.5, normalized=True), crs=PROJECTED_CRS)

# 3. City camera counts: match each block to camera links on the same street within 15 m, average each direction,
#    then add the directions together for a two-way volume.
camera_links = camera_adt.dropna(subset=["startlat", "startlon", "endlat", "endlon", "adt"])
camera_links = gpd.GeoDataFrame(
    camera_links[["roadname", "direction", "adt"]],
    geometry=[LineString([(x1, y1), (x2, y2)]) for x1, y1, x2, y2 in
              camera_links[["startlon", "startlat", "endlon", "endlat"]].to_numpy()],
    crs="EPSG:4326").to_crs(PROJECTED_CRS)
camera_links["roadname"] = camera_links["roadname"].astype("string").str.strip().str.upper()
camera_links["geometry"] = camera_links.geometry.buffer(VOLUME_MATCH_DISTANCE_M)
camera_match = gpd.sjoin(midpoints, camera_links, how="inner", predicate="within")
camera_match = camera_match.loc[camera_match["street_nam"].eq(camera_match["roadname"])]
camera_volume = (camera_match.groupby(["segment_id", "direction"])["adt"].mean()
                 .groupby(level="segment_id").sum().rename("camera_adt"))

# 4. IDOT counts: take the nearest IDOT segment within 15 m (most recent count year if tied).
idot_match = gpd.sjoin_nearest(midpoints, idot_aadt[["AADT", "AADT_YR", "geometry"]].to_crs(PROJECTED_CRS),
                               how="inner", max_distance=VOLUME_MATCH_DISTANCE_M, distance_col="idot_distance_m")
idot_volume = (idot_match.sort_values(["segment_id", "idot_distance_m", "AADT_YR"], ascending=[True, True, False])
               .drop_duplicates("segment_id").set_index("segment_id")["AADT"].rename("idot_aadt"))

# 5. Volume for every block: camera count first, then IDOT count, otherwise the median measured volume for the
#    block's road class. A typical urban volume is used only if a class has fewer than 20 measured blocks.
FALLBACK_ADT = {"Lake Shore Drive": 60000, "Arterial": 15000, "Collector": 6000, "Local street": 1000,
                "Lower-level street": 5000, "Other street": 1000}
road = road.join(camera_volume, on="segment_id").join(idot_volume, on="segment_id")
road["adt"] = road["camera_adt"].combine_first(road["idot_aadt"])
road["volume_measured"] = road["adt"].notna()
class_stats = road.loc[road["volume_measured"]].groupby("road_class")["adt"].agg(["median", "size"])
imputed_adt = {cls: class_stats.loc[cls, "median"] if class_stats["size"].get(cls, 0) >= 20 else fallback
               for cls, fallback in FALLBACK_ADT.items()}
road["adt"] = road["adt"].fillna(road["road_class"].map(imputed_adt))

# 6. Vehicle-miles travelled per day and over the whole 2018–2025 study period.
road["length_mi"] = road["length_m"] / METERS_PER_MILE
road["daily_vmt"] = road["adt"] * road["length_mi"]
road["study_vmt"] = road["daily_vmt"] * STUDY_DAYS

# 7. Coverage of measured volumes by road class.
coverage = (road.assign(measured_miles=road["length_mi"].where(road["volume_measured"], 0))
            .groupby("road_class").agg(miles=("length_mi", "sum"), measured_miles=("measured_miles", "sum"),
                                       daily_vmt=("daily_vmt", "sum"), median_adt=("adt", "median")))
coverage["pct_miles_measured"] = coverage["measured_miles"] / coverage["miles"] * 100
coverage["pct_of_city_vmt"] = coverage["daily_vmt"] / coverage["daily_vmt"].sum() * 100
coverage = coverage.sort_values("daily_vmt", ascending=False)

measured_vmt_share = road.loc[road["volume_measured"], "daily_vmt"].sum() / road["daily_vmt"].sum() * 100
print(f"Street blocks in the exposure model: {len(road):,} ({road['length_mi'].sum():,.0f} miles)")
print(f"Blocks with a measured volume: {road['volume_measured'].sum():,} "
      f"({measured_vmt_share:.1f}% of estimated VMT comes from measured volumes)")
print(f"Estimated daily VMT on CPD-covered streets: {road['daily_vmt'].sum() / 1e6:,.1f} million")
print(f"Estimated VMT over {START_YEAR}–{END_YEAR}: {road['study_vmt'].sum() / 1e9:,.1f} billion")
print("Daily volumes assigned to unmeasured blocks:", {k: int(v) for k, v in imputed_adt.items()})
display(coverage.round(1))

Street blocks in the exposure model: 52,767 (4,116 miles)
Blocks with a measured volume: 18,685 (59.0% of estimated VMT comes from measured volumes)
Estimated daily VMT on CPD-covered streets: 27.2 million
Estimated VMT over 2018–2025: 79.4 billion
Daily volumes assigned to unmeasured blocks: {'Lake Shore Drive': 94500, 'Arterial': 15800, 'Collector': 10908, 'Local street': 3300, 'Lower-level street': 14300, 'Other street': 3200}


,miles,measured_miles,daily_vmt,median_adt,pct_miles_measured,pct_of_city_vmt
road_class,,,,,,
Local street,3089.4,361.0,10730785.4,3300.0,11.7,39.5
Arterial,424.3,407.0,7385632.8,15800.0,95.9,27.2
Collector,473.7,438.5,5548286.7,10908.4,92.6,20.4
Lake Shore Drive,34.3,21.7,3057008.0,94500.0,63.4,11.3
Other street,85.5,3.9,296760.2,3200.0,4.5,1.1
Lower-level street,9.1,8.0,142945.2,14300.0,87.7,0.5


### 3.7 Save the prepared data for Notebook 2

All datasets that Notebook 2 needs are saved together in one file, `data/processed/prepared_data.pkl`. A pickle file keeps column types (dates, text IDs) and map geometry exactly as they are here, so Notebook 2 can start straight from the analysis without repeating the downloads and cleaning.

In [28]:
# Save every dataset that Notebook 2 needs in one file.
prepared = {
    "crash_enriched": crash_enriched,      # one row per crash, with person-level outcome flags
    "df_geo": df_geo[["crash_record_id", "crash_date", "latitude", "longitude", "injuries_total",
                      "injuries_fatal", "first_crash_type"]],  # crashes with valid coordinates
    "road": road,                          # street blocks with traffic volume and vehicle-miles travelled
    "midpoints": midpoints,                # the midpoint of each street block
    "idot_aadt": idot_aadt,                # IDOT traffic counts (used to find much busier roads nearby)
    "tnp": tnp,                            # ride-hail trips by area, weekday and hour
    "l_stations": l_stations, "bus_stops": bus_stops,                 # transit activity
    "divvy_stations": divvy_stations, "divvy_hourly": divvy_hourly,   # bike-share activity
    "community_areas": community_areas, "population": population,     # boundaries and residents
}
PREPARED_FILE = PROCESSED_DIR / "prepared_data.pkl"
pd.to_pickle(prepared, PREPARED_FILE)
print(f"Saved {len(prepared)} datasets to {PREPARED_FILE} ({PREPARED_FILE.stat().st_size / 1e6:,.0f} MB)")
print("Next step: open notebooks/02_analysis.ipynb and run all cells.")

Saved 12 datasets to /Users/armon/data/processed/prepared_data.pkl (761 MB)
Next step: open notebooks/02_analysis.ipynb and run all cells.
